# Notebook 23 — feature_engineering_v3

**Purpose:** Build enriched feature set v3 for H2O Driverless AI binary classification.
Full feature engineering pipeline combining behavioral, temporal, network,
cadastral and sectoral risk features derived from academic red flags literature
for public procurement fraud detection.

**Input:**
- `data/gold/dim_fornecedor.parquet`
- `data/gold/fato_contratos.parquet`
- `data/gold/fato_sancoes.parquet`
- `data/silver/silver_pncp/**/*.parquet`
- `data/silver/silver_compras/**/*.parquet`
- `data/silver/silver_identidade/**/*.parquet`
- `data/silver/silver_ceis/data.parquet`
- `data/silver/silver_cnep/data.parquet`

**Output:**
- `data/serving/serving_fornecedor_features_v3.parquet`
- `data/serving/h2o_supplier_risk_dataset_v3.csv`
- `data/serving/h2o_supplier_risk_dataset_v3_train.csv`
- `data/serving/h2o_supplier_risk_dataset_v3_test.csv`

**Feature groups:**
- Group 1  — Temporal acceleration (contract growth 2023 vs 2024)
- Group 2  — Concentration HHI (organ concentration index)
- Group 3  — Value profile (max/avg ratio, high-value contracts)
- Group 4  — Cadastral basic (company age, CNAE division, capital)
- Group A  — Contract modality, duration, retification
- Group B  — Buyer sphere, power, geography
- Group C  — Cadastral advanced (Simples, MEI, CNAE count, capital ratio)
- Group D  — Value anomalies (std, CV, symbolic contracts)
- Group E  — Temporal advanced (contract intervals, weekend signatures)
- Group F  — Supplier-organ network (exclusivity, annual repetition)
- Group G  — Sectoral risk (sanction rate by CNAE/UF/porte)

**Notes:**
- v2 AUCPR baseline: 0.26 (12 features, notebooks 21+22)
- v3 target: improve AUCPR via manual feature engineering
- Reference period: 2023 and 2024 (complete years, reliable coverage)
- Contracts with data_referencia > CURRENT_DATE excluded (121 anomalous rows)
- Grain: one row per supplier_sk
- Target: tem_sancao — no leakage columns
- Group G uses sectoral aggregates only — no individual sanction records

## Dicionário de Features — serving_fornecedor_features_v3

### Identificação e Target

| Coluna | Tipo | Descrição |
|---|---|---|
| `supplier_sk` | ID | Chave surrogate do fornecedor (MD5 hash do CNPJ). Usado como identificador de linha pelo H2O — não é preditor. |
| `tem_sancao` | Target | 1 se o fornecedor possui sanção registrada no CEIS ou CNEP. 0 caso contrário. É a variável que o modelo tenta prever. |

---

### Identidade Base

Características cadastrais do fornecedor extraídas da Receita Federal via `dim_fornecedor`.

| Coluna | Tipo | Descrição | Por que importa |
|---|---|---|---|
| `porte` | Categórica | Porte da empresa: Micro Empresa, Empresa de Pequeno Porte, Demais | EPP tem taxa de sanção 2.8% — mais que o dobro da média geral de 1.2% |
| `natureza_juridica_desc` | Categórica | Natureza jurídica: Sociedade Limitada, Empresa Pública, Cooperativa, etc. | Empresas Públicas e Cooperativas têm perfis de risco distintos |
| `uf` | Categórica | Estado sede do fornecedor | Taxa de sanção varia por estado — sinal geográfico relevante |
| `situacao_cadastral` | Categórica | Situação na Receita Federal: Ativa, Baixada, Inapta, Suspensa, Nula | Fornecedor inapto ou suspenso com contratos ativos é red flag direto |

---

### Group 1 — Aceleração Temporal

Comportamento do fornecedor ao longo do tempo. Fornecedores que aceleram contratos rapidamente antes de serem sancionados é padrão documentado em fraudes de licitação.

| Coluna | Tipo | Descrição | Por que importa |
|---|---|---|---|
| `contratos_2023` | Numérica | Quantidade de contratos com data de referência em 2023 | Base histórica do volume anual |
| `contratos_2024` | Numérica | Quantidade de contratos em 2024 | Ano mais recente completo — referência principal |
| `contratos_2025` | Numérica | Quantidade de contratos em 2025 | Tendência recente — ano ainda em curso |
| `taxa_crescimento_2023_2024` | Numérica | Razão entre contratos_2024 e contratos_2023. Zero quando não havia contrato em 2023 | Crescimento explosivo de contratos é sinal de irregularidade |
| `meses_desde_primeiro_contrato` | Numérica | Meses entre o primeiro contrato registrado e hoje | Fornecedor com muitos contratos em pouco tempo é red flag |
| `meses_desde_ultimo_contrato` | Numérica | Meses entre o último contrato e hoje. 999 para fornecedores sem contrato | Inatividade recente pode indicar encerramento após irregularidade |

---

### Group 2 — Concentração (HHI)

Mede se o fornecedor distribui contratos entre muitos órgãos ou concentra em poucos.
O HHI (Herfindahl-Hirschman Index) é usado em economia para medir concentração de mercado — aqui adaptado para concentração de relacionamento com órgãos públicos.

| Coluna | Tipo | Descrição | Por que importa |
|---|---|---|---|
| `hhi_orgaos` | Numérica | Índice de concentração de contratos por órgão. 1.0 = todos os contratos com um único órgão. Próximo de 0 = muito disperso | Dependência de um único comprador público é red flag documentado na literatura |
| `qtd_orgaos_unicos` | Numérica | Quantidade de órgãos distintos com os quais o fornecedor tem contrato | Fornecedor com muitos órgãos diferentes tem maior exposição e visibilidade |
| `pct_contratos_top1_orgao` | Numérica | Percentual dos contratos concentrados no órgão mais frequente | Concentração acima de 80% num único órgão é sinal de relacionamento suspeito |
| `pct_valor_top1_orgao_valor` | Numérica | Percentual do valor total concentrado no órgão de maior valor | Concentração financeira em um único comprador amplifica o risco |

---

### Group 3 — Perfil de Valor

Analisa o comportamento dos valores dos contratos. Desvios entre o maior contrato e o ticket médio do fornecedor são indicadores de irregularidade.

| Coluna | Tipo | Descrição | Por que importa |
|---|---|---|---|
| `ratio_max_sobre_medio` | Numérica | Razão entre o maior contrato e o valor médio dos contratos do fornecedor | Um contrato muito acima da média própria indica possível superfaturamento |
| `qtd_contratos_alto_valor` | Numérica | Quantidade de contratos com valor acima de R$ 1 milhão | Contratos de alto valor têm maior exposição a irregularidades |
| `qtd_contratos_muito_alto_valor` | Numérica | Quantidade de contratos acima de R$ 10 milhões | Contratos de altíssimo valor exigem maior escrutínio |
| `pct_contratos_pncp` | Numérica | Percentual dos contratos registrados no PNCP (portal mais transparente) | Fornecedores que evitam o PNCP podem estar tentando reduzir visibilidade |
| `pct_contratos_valor_zero` | Numérica | Percentual de contratos com valor igual a zero | Contratos sem valor declarado indicam dados incompletos ou manipulação |

---

### Group 4 — Cadastral Básico

Características estruturais do fornecedor extraídas da `silver_identidade` (Receita Federal).

| Coluna | Tipo | Descrição | Por que importa |
|---|---|---|---|
| `idade_empresa_anos` | Numérica | Anos completos desde a data de início de atividade até hoje | Empresas muito jovens com alto volume de contratos são red flag clássico |
| `cnae_divisao` | Categórica | Primeiros 2 dígitos do CNAE fiscal principal — representa o setor econômico (87 divisões possíveis) | Alguns setores têm taxa de sanção 8x acima da média — sinal setorial relevante |
| `capital_social` | Numérica | Capital social registrado na Receita Federal em reais | Capital muito baixo para o volume contratado é indicador de empresa de fachada |
| `empresa_jovem` | Binária | 1 se a empresa tem menos de 2 anos de existência | Empresas recentes com muitos contratos públicos merecem atenção especial |

---

### Group A — Perfil Contratual Avançado

Detalhes sobre modalidade, duração e alterações dos contratos. Derivado de `silver_compras` e `silver_pncp`.

| Coluna | Tipo | Descrição | Por que importa |
|---|---|---|---|
| `pct_dispensa` | Numérica | Percentual dos contratos firmados por dispensa de licitação | Dispensa é modalidade que dispensa concorrência — concentração nela é red flag documentado |
| `pct_pregao` | Numérica | Percentual firmado por pregão — modalidade mais transparente e competitiva | Alto percentual de pregão indica maior conformidade com regras de concorrência |
| `pct_inexigibilidade` | Numérica | Percentual firmado por inexigibilidade — fornecedor único declarado | Inexigibilidade legítima existe, mas concentração nela merece investigação |
| `duracao_media_dias` | Numérica | Duração média dos contratos em dias (entre vigência inicial e final) | Contratos muito curtos em grande volume ou contratos excessivamente longos são anomalias |
| `duracao_max_dias` | Numérica | Duração do contrato mais longo em dias | Contratos com vigência de décadas são red flag de superfaturamento ou vínculos impróprios |
| `pct_empenho` | Numérica | Percentual dos registros PNCP classificados como empenho | Empenho é forma mais simples de contratação — concentração pode indicar evasão de controle |
| `pct_contrato_formal` | Numérica | Percentual dos registros PNCP classificados como contrato formal (termo inicial) | Alto percentual de contratos formais indica maior rigor no relacionamento com o poder público |
| `qtd_retificacoes` | Numérica | Total de retificações realizadas em contratos PNCP | Retificações frequentes indicam alterações posteriores — red flag documentado na literatura acadêmica |
| `tem_retificacao` | Binária | 1 se o fornecedor tem ao menos uma retificação registrada | Presença de qualquer retificação já é sinal que merece atenção |

---

### Group B — Esfera, Poder e Geografia do Comprador

Perfil dos órgãos compradores do fornecedor. Derivado de `silver_pncp`.

| Coluna | Tipo | Descrição | Por que importa |
|---|---|---|---|
| `pct_esfera_federal` | Numérica | Percentual dos contratos com órgãos federais | Órgãos federais têm maior controle — concentração pode indicar perfil de risco distinto |
| `pct_esfera_estadual` | Numérica | Percentual com órgãos estaduais | Controle varia por estado — relevante para análise regional |
| `pct_esfera_municipal` | Numérica | Percentual com órgãos municipais | Municípios pequenos têm menor capacidade de fiscalização — risco historicamente mais alto |
| `pct_poder_executivo` | Numérica | Percentual com órgãos do poder executivo | Maior volume de contratos — referência base |
| `pct_poder_legislativo` | Numérica | Percentual com órgãos do legislativo | Contratos com câmaras municipais e assembleias merecem atenção especial |
| `pct_poder_judiciario` | Numérica | Percentual com órgãos do judiciário | Perfil de fornecedor muito distinto dos demais poderes |
| `qtd_ufs_compradoras` | Numérica | Quantidade de estados distintos onde o fornecedor tem compradores | Dispersão geográfica alta pode indicar empresa com atuação nacional ou pulverização suspeita |
| `tem_comprador_externo` | Binária | 1 se o fornecedor tem contratos com órgãos fora de seu estado de sede | Fornecedor local contratado por órgãos de outros estados é comportamento a investigar |

---

### Group C — Cadastral Avançado

Características tributárias e de diversificação de atividade. Derivado de `silver_identidade`.

| Coluna | Tipo | Descrição | Por que importa |
|---|---|---|---|
| `is_simples` | Binária | 1 se optante pelo Simples Nacional | Empresas do Simples têm benefícios tributários — cruzamento com volume contratual é relevante |
| `is_mei` | Binária | 1 se é Microempreendedor Individual | MEI tem limite de faturamento anual de R$ 81 mil — contratos acima disso são irregulares por lei |
| `qtd_cnaes_secundarios` | Numérica | Quantidade de CNAEs secundários declarados | Empresas com muitas atividades secundárias podem estar tentando se enquadrar em editais de áreas diversas |
| `ratio_valor_sobre_capital` | Numérica | Razão entre o valor total contratado e o capital social registrado | Empresa com R$ 1.000 de capital e R$ 10 milhões em contratos é red flag direto de empresa de fachada |

---

### Group D — Anomalias de Valor

Métricas estatísticas que identificam comportamentos anômalos nos valores dos contratos.

| Coluna | Tipo | Descrição | Por que importa |
|---|---|---|---|
| `std_valor_contratos` | Numérica | Desvio padrão dos valores de contrato do fornecedor | Alta variância nos valores indica portfólio irregular — mistura de contratos ínfimos e gigantes |
| `cv_valor_contratos` | Numérica | Coeficiente de variação (desvio padrão / média) — normalizado por escala | CV acima de 2.0 indica que os valores são extremamente dispersos em relação à própria média do fornecedor |
| `qtd_contratos_valor_1real` | Numérica | Quantidade de contratos com valor exatamente R$ 1,00 | Contratos de R$ 1,00 são conhecidos na literatura como placeholder de dados incompletos ou manipulação intencional de valor |
| `gap_max_vs_medio` | Numérica | Diferença absoluta entre o maior contrato e o valor médio | Distância muito grande entre o maior contrato e a média indica outlier suspeito no portfólio |

---

### Group E — Temporal Avançado

Padrões temporais detalhados extraídos das datas de assinatura dos contratos PNCP.

| Coluna | Tipo | Descrição | Por que importa |
|---|---|---|---|
| `meses_entre_contratos_avg` | Numérica | Intervalo médio em dias entre contratos consecutivos do fornecedor | Fornecedores com contratos muito espaçados têm perfil diferente dos com contratos contínuos |
| `dias_entre_contratos_min` | Numérica | Menor intervalo entre dois contratos consecutivos | Intervalo zero indica múltiplos contratos no mesmo dia — rajada de contratos é red flag |
| `qtd_contratos_fds` | Numérica | Quantidade de contratos assinados em fim de semana (sábado ou domingo) | Contratos assinados em fim de semana fogem do fluxo normal de trabalho — podem indicar urgência artificial |
| `pct_contratos_fds` | Numérica | Percentual dos contratos assinados em fim de semana | 20% da base tem contratos em fim de semana — percentual acima disso é anomalia |
| `max_contratos_mesmo_mes` | Numérica | Maior concentração de contratos em um único mês | Pico de 2.305 contratos num mês único foi observado na base — rajadas mensais são red flag |

---

### Group F — Rede Fornecedor-Órgão

Analisa o padrão de relacionamento entre o fornecedor e os órgãos compradores ao longo do tempo.

| Coluna | Tipo | Descrição | Por que importa |
|---|---|---|---|
| `fornecedor_exclusivo_orgao` | Binária | 1 se 100% dos contratos são com um único órgão | Exclusividade total com um órgão é red flag de relação imprópria — 55.176 casos na base |
| `qtd_orgaos_distintos_total` | Numérica | Total de órgãos distintos com os quais o fornecedor já contratou | Visão completa da rede de relacionamentos governamentais do fornecedor |
| `max_anos_mesmo_orgao` | Numérica | Maior número de anos consecutivos contratando com o mesmo órgão | Relacionamentos de longa duração com o mesmo órgão merecem análise de dependência |
| `repete_mesmo_orgao_3anos` | Binária | 1 se o fornecedor contratou com o mesmo órgão em 3 ou mais anos distintos | Repetição sistemática com o mesmo comprador ao longo de anos é sinal de vínculo suspeito — 9.641 casos |

---

### Group G — Risco Setorial

Taxa de sanção calculada por setor, estado e porte — agrupada, nunca individual.
Não há leakage: o target individual (`tem_sancao`) não é usado para calcular o risco do próprio fornecedor — apenas o comportamento agregado do setor.

| Coluna | Tipo | Descrição | Por que importa |
|---|---|---|---|
| `taxa_sancao_cnae` | Numérica | Percentual de fornecedores sancionados que atuam na mesma divisão CNAE do fornecedor avaliado | Alguns setores têm taxa de 10.2% — 8x acima da média geral de 1.2%. Pertencer a um setor de alto risco é sinal relevante |
| `taxa_sancao_uf` | Numérica | Percentual de fornecedores sancionados no mesmo estado do fornecedor | Taxa varia por estado — contexto regional de fiscalização e controle é diferente |
| `taxa_sancao_porte` | Numérica | Percentual de fornecedores sancionados com o mesmo porte | EPP tem taxa 2.8% vs 0.54% para Demais — o porte já carrega risco diferenciado |

In [ ]:
## Step 1 — Imports and configuration

import duckdb
from pathlib import Path
import sys

ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "utils" / "paths.py").exists():
        ROOT = candidate
        break
sys.path.insert(0, str(ROOT))

from utils.paths import (
    get_project_root, gold_path, silver_path,
    serving_path, to_sql_path, ensure_dir
)

ROOT = get_project_root()

# ── paths ─────────────────────────────────────────────────────────────────────
DIM_FORNECEDOR_PATH    = gold_path(ROOT, "dim_fornecedor")
FATO_CONTRATOS_PATH    = gold_path(ROOT, "fato_contratos")
FATO_SANCOES_PATH      = gold_path(ROOT, "fato_sancoes")
SILVER_PNCP_PATH       = silver_path(ROOT, "silver_pncp",       glob=True)
SILVER_COMPRAS_PATH    = silver_path(ROOT, "silver_compras",    glob=True)
SILVER_IDENTIDADE_PATH = silver_path(ROOT, "silver_identidade", glob=True)
SILVER_CEIS_PATH       = silver_path(ROOT, "silver_ceis",       glob=False)
SILVER_CNEP_PATH       = silver_path(ROOT, "silver_cnep",       glob=False)
OUTPUT_PARQUET         = serving_path(ROOT, "serving_fornecedor_features_v3")
OUTPUT_CSV             = to_sql_path(ROOT / "data" / "serving" / "h2o_supplier_risk_dataset_v3.csv")

ensure_dir(ROOT / "data" / "serving")

con = duckdb.connect()

print(f"{'Source':<25} {'Exists':<8} Path")
print("-" * 90)
paths = {
    "dim_fornecedor":    DIM_FORNECEDOR_PATH,
    "fato_contratos":    FATO_CONTRATOS_PATH,
    "fato_sancoes":      FATO_SANCOES_PATH,
    "silver_pncp":       SILVER_PNCP_PATH,
    "silver_compras":    SILVER_COMPRAS_PATH,
    "silver_identidade": SILVER_IDENTIDADE_PATH,
    "silver_ceis":       SILVER_CEIS_PATH,
    "silver_cnep":       SILVER_CNEP_PATH,
}
for name, path in paths.items():
    exists = Path(path.replace("**/*.parquet", "")).exists()
    print(f"{name:<25} {str(exists):<8} {path}")

print(f"\nOUTPUT PARQUET: {OUTPUT_PARQUET}")
print(f"OUTPUT CSV    : {OUTPUT_CSV}")

In [ ]:
## Step 2 — Load sources

for name, path in paths.items():
    con.execute(f"""
        CREATE OR REPLACE VIEW {name} AS
        SELECT * FROM read_parquet('{path}')
    """)
    n = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"{name:<25}: {n:>12,} rows")

In [ ]:
## Step 3 — Supplier base

con.execute("""
    CREATE OR REPLACE TABLE supplier_base AS
    SELECT
        supplier_sk,
        cnpj_normalized,
        porte,
        natureza_juridica_desc,
        uf,
        situacao_cadastral,
        tem_sancao
    FROM dim_fornecedor
    WHERE is_current = true
""")

n = con.execute("SELECT COUNT(*) FROM supplier_base").fetchone()[0]
print(f"supplier_base: {n:,} rows")

In [ ]:
## Step 4 — Group 1: Temporal acceleration features
# Reference period: 2023 vs 2024 (complete years with reliable coverage)
# Excludes future dates (data_referencia > CURRENT_DATE)

con.execute("""
    CREATE OR REPLACE TABLE feat_g1_temporal AS
    SELECT
        supplier_sk,

        COUNT(*) FILTER (WHERE YEAR(data_referencia) = 2023) AS contratos_2023,
        COUNT(*) FILTER (WHERE YEAR(data_referencia) = 2024) AS contratos_2024,
        COUNT(*) FILTER (WHERE YEAR(data_referencia) = 2025) AS contratos_2025,

        -- taxa de crescimento 2023 → 2024
        ROUND(
            (COUNT(*) FILTER (WHERE YEAR(data_referencia) = 2024) -
             COUNT(*) FILTER (WHERE YEAR(data_referencia) = 2023))
            * 1.0 /
            NULLIF(COUNT(*) FILTER (WHERE YEAR(data_referencia) = 2023), 0)
        , 4) AS taxa_crescimento_2023_2024,

        -- span de atividade
        DATEDIFF('month', MIN(data_referencia), CURRENT_DATE) AS meses_desde_primeiro_contrato,
        DATEDIFF('month', MAX(data_referencia), CURRENT_DATE) AS meses_desde_ultimo_contrato

    FROM fato_contratos
    WHERE data_referencia IS NOT NULL
      AND data_referencia <= CURRENT_DATE
    GROUP BY supplier_sk
""")

n = con.execute("SELECT COUNT(*) FROM feat_g1_temporal").fetchone()[0]
print(f"feat_g1_temporal: {n:,} rows")

In [ ]:
## Step 5 — Group 2: Concentration features (HHI)
# HHI = 1.0 → all contracts with single organ (maximum concentration)
# HHI → 0   → contracts spread across many organs (maximum dispersion)

con.execute("""
    CREATE OR REPLACE TABLE feat_g2_concentracao AS
    WITH organ_shares AS (
        SELECT
            supplier_sk,
            codigo_orgao,
            COUNT(*)                                          AS qtd_orgao,
            SUM(COUNT(*)) OVER (PARTITION BY supplier_sk)    AS qtd_total
        FROM fato_contratos
        WHERE data_referencia IS NOT NULL
          AND data_referencia <= CURRENT_DATE
          AND codigo_orgao IS NOT NULL
        GROUP BY supplier_sk, codigo_orgao
    ),
    hhi_calc AS (
        SELECT
            supplier_sk,
            ROUND(SUM(POWER(qtd_orgao * 1.0 / qtd_total, 2)), 4) AS hhi_orgaos,
            COUNT(DISTINCT codigo_orgao)                          AS qtd_orgaos_unicos,
            MAX(qtd_orgao * 1.0 / qtd_total)                     AS pct_contratos_top1_orgao
        FROM organ_shares
        GROUP BY supplier_sk
    ),
    valor_shares AS (
        SELECT
            supplier_sk,
            ROUND(
                MAX(valor_por_orgao) / NULLIF(SUM(valor_por_orgao), 0)
            , 4) AS pct_valor_top1_orgao_valor
        FROM (
            SELECT
                supplier_sk,
                codigo_orgao,
                SUM(valor_inicial) FILTER (WHERE NOT _valor_negativo) AS valor_por_orgao
            FROM fato_contratos
            WHERE data_referencia IS NOT NULL
              AND data_referencia <= CURRENT_DATE
              AND codigo_orgao IS NOT NULL
            GROUP BY supplier_sk, codigo_orgao
        )
        GROUP BY supplier_sk
    )
    SELECT
        h.supplier_sk,
        h.hhi_orgaos,
        h.qtd_orgaos_unicos,
        h.pct_contratos_top1_orgao,
        v.pct_valor_top1_orgao_valor
    FROM hhi_calc h
    LEFT JOIN valor_shares v USING (supplier_sk)
""")

n = con.execute("SELECT COUNT(*) FROM feat_g2_concentracao").fetchone()[0]
print(f"feat_g2_concentracao: {n:,} rows")

In [ ]:
## Step 6 — Group 3: Value profile features

con.execute("""
    CREATE OR REPLACE TABLE feat_g3_valor AS
    SELECT
        supplier_sk,

        ROUND(
            MAX(valor_inicial) FILTER (WHERE NOT _valor_negativo) /
            NULLIF(AVG(valor_inicial) FILTER (WHERE NOT _valor_negativo), 0)
        , 2) AS ratio_max_sobre_medio,

        COUNT(*) FILTER (
            WHERE NOT _valor_negativo AND valor_inicial >= 1000000
        )                                           AS qtd_contratos_alto_valor,

        COUNT(*) FILTER (
            WHERE NOT _valor_negativo AND valor_inicial >= 10000000
        )                                           AS qtd_contratos_muito_alto_valor,

        ROUND(
            COUNT(*) FILTER (WHERE fonte = 'pncp') * 1.0 /
            NULLIF(COUNT(*), 0)
        , 4)                                        AS pct_contratos_pncp,

        ROUND(
            COUNT(*) FILTER (WHERE valor_inicial = 0) * 1.0 /
            NULLIF(COUNT(*), 0)
        , 4)                                        AS pct_contratos_valor_zero

    FROM fato_contratos
    WHERE data_referencia IS NOT NULL
      AND data_referencia <= CURRENT_DATE
    GROUP BY supplier_sk
""")

n = con.execute("SELECT COUNT(*) FROM feat_g3_valor").fetchone()[0]
print(f"feat_g3_valor: {n:,} rows")

In [ ]:
## Step 7 — Group 4: Cadastral basic features (silver_identidade)

con.execute("""
    CREATE OR REPLACE TABLE feat_g4_cadastral AS
    SELECT
        b.supplier_sk,

        DATEDIFF('year',
            s.data_inicio_atividade,
            CURRENT_DATE)                           AS idade_empresa_anos,

        LEFT(s.cnae_fiscal_principal_cod, 2)        AS cnae_divisao,

        CAST(s.capital_social AS DOUBLE)            AS capital_social,

        CASE
            WHEN DATEDIFF('year', s.data_inicio_atividade, CURRENT_DATE) < 2
            THEN 1 ELSE 0
        END                                         AS empresa_jovem

    FROM supplier_base b
    INNER JOIN silver_identidade s
        ON b.cnpj_normalized = s.cnpj_normalized
    WHERE s.cnpj_normalized IS NOT NULL
""")

n = con.execute("SELECT COUNT(*) FROM feat_g4_cadastral").fetchone()[0]
print(f"feat_g4_cadastral: {n:,} rows")

In [ ]:
## Step 8 — Group A: Contract modality, duration and retification
# silver_compras: modalidade, categoria, duração de vigência
# silver_pncp:    tipo_contrato, retificações

con.execute("""
    CREATE OR REPLACE TABLE feat_gA_contrato AS
    WITH compras_agg AS (
        SELECT
            cnpj_normalized,

            -- modalidade
            ROUND(COUNT(*) FILTER (WHERE nome_modalidade = 'Dispensa') * 1.0
                / NULLIF(COUNT(*), 0), 4)           AS pct_dispensa,
            ROUND(COUNT(*) FILTER (WHERE nome_modalidade = 'Pregão') * 1.0
                / NULLIF(COUNT(*), 0), 4)           AS pct_pregao,
            ROUND(COUNT(*) FILTER (WHERE nome_modalidade = 'Inexigibilidade') * 1.0
                / NULLIF(COUNT(*), 0), 4)           AS pct_inexigibilidade,

            -- duração de vigência em dias
            ROUND(AVG(
                DATEDIFF('day', data_vigencia_inicial, data_vigencia_final)
            ) FILTER (
                WHERE data_vigencia_inicial IS NOT NULL
                  AND data_vigencia_final IS NOT NULL
                  AND data_vigencia_final >= data_vigencia_inicial
            ), 1)                                   AS duracao_media_dias,

            MAX(
                DATEDIFF('day', data_vigencia_inicial, data_vigencia_final)
            ) FILTER (
                WHERE data_vigencia_inicial IS NOT NULL
                  AND data_vigencia_final IS NOT NULL
                  AND data_vigencia_final >= data_vigencia_inicial
            )                                       AS duracao_max_dias

        FROM silver_compras
        GROUP BY cnpj_normalized
    ),
    pncp_agg AS (
        SELECT
            cnpj_normalized,

            -- tipo de contrato
            ROUND(COUNT(*) FILTER (WHERE tipo_contrato_nome = 'Empenho') * 1.0
                / NULLIF(COUNT(*), 0), 4)           AS pct_empenho,
            ROUND(COUNT(*) FILTER (
                WHERE tipo_contrato_nome = 'Contrato (termo inicial)'
            ) * 1.0 / NULLIF(COUNT(*), 0), 4)      AS pct_contrato_formal,

            -- retificações
            SUM(TRY_CAST(numero_retificacao AS INTEGER))
                                                    AS qtd_retificacoes,
            MAX(CASE
                WHEN TRY_CAST(numero_retificacao AS INTEGER) > 0
                THEN 1 ELSE 0
            END)                                    AS tem_retificacao

        FROM silver_pncp
        WHERE cnpj_normalized IS NOT NULL
        GROUP BY cnpj_normalized
    )
    SELECT
        b.supplier_sk,
        COALESCE(c.pct_dispensa,        0.0)        AS pct_dispensa,
        COALESCE(c.pct_pregao,          0.0)        AS pct_pregao,
        COALESCE(c.pct_inexigibilidade, 0.0)        AS pct_inexigibilidade,
        COALESCE(c.duracao_media_dias,  0.0)        AS duracao_media_dias,
        COALESCE(c.duracao_max_dias,    0)          AS duracao_max_dias,
        COALESCE(p.pct_empenho,         0.0)        AS pct_empenho,
        COALESCE(p.pct_contrato_formal, 0.0)        AS pct_contrato_formal,
        COALESCE(p.qtd_retificacoes,    0)          AS qtd_retificacoes,
        COALESCE(p.tem_retificacao,     0)          AS tem_retificacao

    FROM supplier_base b
    LEFT JOIN compras_agg c ON b.cnpj_normalized = c.cnpj_normalized
    LEFT JOIN pncp_agg    p ON b.cnpj_normalized = p.cnpj_normalized
""")

n = con.execute("SELECT COUNT(*) FROM feat_gA_contrato").fetchone()[0]
print(f"feat_gA_contrato: {n:,} rows")

print(con.execute("""
    SELECT
        ROUND(AVG(pct_dispensa), 3)         AS avg_pct_dispensa,
        ROUND(AVG(pct_pregao), 3)           AS avg_pct_pregao,
        ROUND(AVG(duracao_media_dias), 1)   AS avg_duracao_dias,
        SUM(tem_retificacao)                AS com_retificacao,
        SUM(qtd_retificacoes)               AS total_retificacoes
    FROM feat_gA_contrato
""").df().to_string(index=False))

In [ ]:
## Step 9 — Group B: Buyer sphere, power and geography
# orgao_esfera: F=Federal, E=Estadual, M=Municipal, N=Nacional, D=Distrital
# orgao_poder:  E=Executivo, L=Legislativo, J=Judiciário, N=Não definido
# unidade_uf:   UF da unidade compradora

con.execute("""
    CREATE OR REPLACE TABLE feat_gB_esfera AS
    WITH pncp_agg AS (
        SELECT
            cnpj_normalized,

            -- esfera do órgão comprador
            ROUND(COUNT(*) FILTER (WHERE orgao_esfera = 'F') * 1.0
                / NULLIF(COUNT(*), 0), 4)           AS pct_esfera_federal,
            ROUND(COUNT(*) FILTER (WHERE orgao_esfera = 'E') * 1.0
                / NULLIF(COUNT(*), 0), 4)           AS pct_esfera_estadual,
            ROUND(COUNT(*) FILTER (WHERE orgao_esfera = 'M') * 1.0
                / NULLIF(COUNT(*), 0), 4)           AS pct_esfera_municipal,

            -- poder do órgão comprador
            ROUND(COUNT(*) FILTER (WHERE orgao_poder = 'E') * 1.0
                / NULLIF(COUNT(*), 0), 4)           AS pct_poder_executivo,
            ROUND(COUNT(*) FILTER (WHERE orgao_poder = 'L') * 1.0
                / NULLIF(COUNT(*), 0), 4)           AS pct_poder_legislativo,
            ROUND(COUNT(*) FILTER (WHERE orgao_poder = 'J') * 1.0
                / NULLIF(COUNT(*), 0), 4)           AS pct_poder_judiciario,

            -- dispersão geográfica dos compradores
            COUNT(DISTINCT unidade_uf)              AS qtd_ufs_compradoras,

            -- fornece fora do próprio estado
            MAX(CASE
                WHEN unidade_uf IS NOT NULL
                 AND unidade_uf != ''
                THEN 1 ELSE 0
            END)                                    AS tem_comprador_externo

        FROM silver_pncp
        WHERE cnpj_normalized IS NOT NULL
        GROUP BY cnpj_normalized
    )
    SELECT
        b.supplier_sk,
        COALESCE(p.pct_esfera_federal,    0.0)  AS pct_esfera_federal,
        COALESCE(p.pct_esfera_estadual,   0.0)  AS pct_esfera_estadual,
        COALESCE(p.pct_esfera_municipal,  0.0)  AS pct_esfera_municipal,
        COALESCE(p.pct_poder_executivo,   0.0)  AS pct_poder_executivo,
        COALESCE(p.pct_poder_legislativo, 0.0)  AS pct_poder_legislativo,
        COALESCE(p.pct_poder_judiciario,  0.0)  AS pct_poder_judiciario,
        COALESCE(p.qtd_ufs_compradoras,   0)    AS qtd_ufs_compradoras,
        COALESCE(p.tem_comprador_externo, 0)    AS tem_comprador_externo

    FROM supplier_base b
    LEFT JOIN pncp_agg p ON b.cnpj_normalized = p.cnpj_normalized
""")

n = con.execute("SELECT COUNT(*) FROM feat_gB_esfera").fetchone()[0]
print(f"feat_gB_esfera: {n:,} rows")

print(con.execute("""
    SELECT
        ROUND(AVG(pct_esfera_federal), 3)   AS avg_pct_federal,
        ROUND(AVG(pct_esfera_municipal), 3) AS avg_pct_municipal,
        ROUND(AVG(qtd_ufs_compradoras), 2)  AS avg_ufs_compradoras,
        SUM(tem_comprador_externo)          AS com_comprador_externo
    FROM feat_gB_esfera
""").df().to_string(index=False))

In [ ]:
## Step 10 — Group C: Cadastral advanced features

con.execute("""
    CREATE OR REPLACE TABLE feat_gC_cadastral_adv AS
    WITH identidade_agg AS (
        SELECT
            cnpj_normalized,

            CASE WHEN opcao_simples = true THEN 1 ELSE 0 END    AS is_simples,
            CASE WHEN opcao_mei     = true THEN 1 ELSE 0 END    AS is_mei,

            CASE
                WHEN cnae_fiscal_secundaria IS NULL
                  OR cnae_fiscal_secundaria = ''
                THEN 0
                ELSE ARRAY_LENGTH(
                    STRING_SPLIT(cnae_fiscal_secundaria, ',')
                )
            END                                                  AS qtd_cnaes_secundarios,

            CAST(capital_social AS DOUBLE)                       AS capital_social_raw

        FROM silver_identidade
    ),
    contratos_valor AS (
        SELECT
            cnpj_normalized,
            SUM(CASE WHEN valor_inicial > 0 THEN valor_inicial ELSE 0 END) AS valor_total_raw
        FROM silver_pncp
        WHERE cnpj_normalized IS NOT NULL
        GROUP BY cnpj_normalized
    )
    SELECT
        b.supplier_sk,
        COALESCE(i.is_simples,             0)       AS is_simples,
        COALESCE(i.is_mei,                 0)       AS is_mei,
        COALESCE(i.qtd_cnaes_secundarios,  0)       AS qtd_cnaes_secundarios,

        ROUND(
            COALESCE(v.valor_total_raw, 0) /
            NULLIF(COALESCE(i.capital_social_raw, 0), 0)
        , 2)                                        AS ratio_valor_sobre_capital

    FROM supplier_base b
    LEFT JOIN identidade_agg  i ON b.cnpj_normalized = i.cnpj_normalized
    LEFT JOIN contratos_valor v ON b.cnpj_normalized = v.cnpj_normalized
""")

n = con.execute("SELECT COUNT(*) FROM feat_gC_cadastral_adv").fetchone()[0]
print(f"feat_gC_cadastral_adv: {n:,} rows")

print(con.execute("""
    SELECT
        SUM(is_simples)                                         AS total_simples,
        SUM(is_mei)                                             AS total_mei,
        ROUND(AVG(qtd_cnaes_secundarios), 1)                    AS avg_cnaes_secundarios,
        COUNT(*) FILTER (WHERE ratio_valor_sobre_capital > 100) AS ratio_acima_100x
    FROM feat_gC_cadastral_adv
""").df().to_string(index=False))

In [ ]:
## Step 11 — Group D: Value anomaly features
# std e CV medem dispersão dos valores — alta variância é red flag
# contratos simbólicos (R$ 1,00) indicam dados incompletos ou manipulação

con.execute("""
    CREATE OR REPLACE TABLE feat_gD_anomalia_valor AS
    SELECT
        supplier_sk,

        -- desvio padrão dos valores de contrato
        ROUND(STDDEV(valor_inicial) FILTER (
            WHERE NOT _valor_negativo AND valor_inicial > 0
        ), 2)                                           AS std_valor_contratos,

        -- coeficiente de variação (std / mean) — normaliza por escala
        ROUND(
            STDDEV(valor_inicial) FILTER (WHERE NOT _valor_negativo AND valor_inicial > 0) /
            NULLIF(AVG(valor_inicial) FILTER (WHERE NOT _valor_negativo AND valor_inicial > 0), 0)
        , 4)                                            AS cv_valor_contratos,

        -- contratos com valor simbólico (R$ 1,00)
        COUNT(*) FILTER (WHERE valor_inicial = 1.0)     AS qtd_contratos_valor_1real,

        -- discrepância entre valor_inicial e valor parcela (silver_pncp não disponível aqui)
        -- usamos fato_contratos apenas
        ROUND(
            MAX(valor_inicial) FILTER (WHERE NOT _valor_negativo) -
            AVG(valor_inicial) FILTER (WHERE NOT _valor_negativo)
        , 2)                                            AS gap_max_vs_medio

    FROM fato_contratos
    WHERE data_referencia IS NOT NULL
      AND data_referencia <= CURRENT_DATE
    GROUP BY supplier_sk
""")

n = con.execute("SELECT COUNT(*) FROM feat_gD_anomalia_valor").fetchone()[0]
print(f"feat_gD_anomalia_valor: {n:,} rows")

print(con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE cv_valor_contratos > 2)      AS cv_acima_2x,
        SUM(qtd_contratos_valor_1real)                      AS total_contratos_1real,
        COUNT(*) FILTER (WHERE qtd_contratos_valor_1real > 0) AS fornecedores_com_1real
    FROM feat_gD_anomalia_valor
""").df().to_string(index=False))

In [ ]:
## Step 12 — Group E: Advanced temporal features
# Intervalo entre contratos consecutivos — rajadas de contratos são red flag
# Contratos assinados em fim de semana (data_assinatura da silver_pncp)

con.execute("""
    CREATE OR REPLACE TABLE feat_gE_temporal_adv AS
    WITH contratos_ordenados AS (
        SELECT
            cnpj_normalized,
            data_assinatura,
            LAG(data_assinatura) OVER (
                PARTITION BY cnpj_normalized
                ORDER BY data_assinatura
            )                                       AS data_anterior
        FROM silver_pncp
        WHERE data_assinatura IS NOT NULL
          AND data_assinatura <= CURRENT_DATE
          AND cnpj_normalized IS NOT NULL
    ),
    intervalos AS (
        SELECT
            cnpj_normalized,
            DATEDIFF('day', data_anterior, data_assinatura) AS intervalo_dias
        FROM contratos_ordenados
        WHERE data_anterior IS NOT NULL
          AND DATEDIFF('day', data_anterior, data_assinatura) >= 0
    ),
    intervalo_agg AS (
        SELECT
            cnpj_normalized,
            ROUND(AVG(intervalo_dias), 1)   AS meses_entre_contratos_avg,
            MIN(intervalo_dias)             AS dias_entre_contratos_min
        FROM intervalos
        GROUP BY cnpj_normalized
    ),
    fim_semana_agg AS (
        SELECT
            cnpj_normalized,
            COUNT(*) FILTER (
                WHERE DAYOFWEEK(data_assinatura) IN (1, 7)
            )                                       AS qtd_contratos_fds,
            ROUND(
                COUNT(*) FILTER (WHERE DAYOFWEEK(data_assinatura) IN (1, 7)) * 1.0 /
                NULLIF(COUNT(*), 0)
            , 4)                                    AS pct_contratos_fds,
            -- concentração mensal máxima
            MAX(contratos_no_mes)                   AS max_contratos_mesmo_mes
        FROM (
            SELECT
                cnpj_normalized,
                data_assinatura,
                COUNT(*) OVER (
                    PARTITION BY cnpj_normalized,
                    DATE_TRUNC('month', data_assinatura)
                )                                   AS contratos_no_mes
            FROM silver_pncp
            WHERE data_assinatura IS NOT NULL
              AND data_assinatura <= CURRENT_DATE
              AND cnpj_normalized IS NOT NULL
        )
        GROUP BY cnpj_normalized
    )
    SELECT
        b.supplier_sk,
        COALESCE(i.meses_entre_contratos_avg,  0.0) AS meses_entre_contratos_avg,
        COALESCE(i.dias_entre_contratos_min,   0)   AS dias_entre_contratos_min,
        COALESCE(f.qtd_contratos_fds,          0)   AS qtd_contratos_fds,
        COALESCE(f.pct_contratos_fds,          0.0) AS pct_contratos_fds,
        COALESCE(f.max_contratos_mesmo_mes,    0)   AS max_contratos_mesmo_mes

    FROM supplier_base b
    LEFT JOIN intervalo_agg  i ON b.cnpj_normalized = i.cnpj_normalized
    LEFT JOIN fim_semana_agg f ON b.cnpj_normalized = f.cnpj_normalized
""")

n = con.execute("SELECT COUNT(*) FROM feat_gE_temporal_adv").fetchone()[0]
print(f"feat_gE_temporal_adv: {n:,} rows")

print(con.execute("""
    SELECT
        ROUND(AVG(meses_entre_contratos_avg), 1)        AS avg_intervalo_dias,
        COUNT(*) FILTER (WHERE dias_entre_contratos_min = 0) AS contratos_mesmo_dia,
        ROUND(AVG(pct_contratos_fds), 3)                AS avg_pct_fds,
        MAX(max_contratos_mesmo_mes)                    AS max_contratos_mes
    FROM feat_gE_temporal_adv
""").df().to_string(index=False))

In [ ]:
## Step 13 — Group F: Supplier-organ network features
# Exclusividade: fornecedor que concentra 100% dos contratos em um único órgão
# Repetição anual: contratado pelo mesmo órgão em 3+ anos consecutivos

con.execute("""
    CREATE OR REPLACE TABLE feat_gF_rede AS
    WITH organ_year AS (
        SELECT
            supplier_sk,
            codigo_orgao,
            YEAR(data_referencia)               AS ano
        FROM fato_contratos
        WHERE data_referencia IS NOT NULL
          AND data_referencia <= CURRENT_DATE
          AND codigo_orgao IS NOT NULL
        GROUP BY supplier_sk, codigo_orgao, YEAR(data_referencia)
    ),
    organ_years_count AS (
        SELECT
            supplier_sk,
            codigo_orgao,
            COUNT(DISTINCT ano)                 AS anos_com_orgao
        FROM organ_year
        GROUP BY supplier_sk, codigo_orgao
    ),
    repeticao_agg AS (
        SELECT
            supplier_sk,
            MAX(anos_com_orgao)                 AS max_anos_mesmo_orgao,
            MAX(CASE WHEN anos_com_orgao >= 3 THEN 1 ELSE 0 END)
                                                AS repete_mesmo_orgao_3anos
        FROM organ_years_count
        GROUP BY supplier_sk
    ),
    exclusividade AS (
        SELECT
            supplier_sk,
            CASE WHEN COUNT(DISTINCT codigo_orgao) = 1
                THEN 1 ELSE 0
            END                                 AS fornecedor_exclusivo_orgao,
            COUNT(DISTINCT codigo_orgao)        AS qtd_orgaos_distintos_total
        FROM fato_contratos
        WHERE data_referencia IS NOT NULL
          AND data_referencia <= CURRENT_DATE
          AND codigo_orgao IS NOT NULL
        GROUP BY supplier_sk
    )
    SELECT
        b.supplier_sk,
        COALESCE(e.fornecedor_exclusivo_orgao,  0)  AS fornecedor_exclusivo_orgao,
        COALESCE(e.qtd_orgaos_distintos_total,  0)  AS qtd_orgaos_distintos_total,
        COALESCE(r.max_anos_mesmo_orgao,        0)  AS max_anos_mesmo_orgao,
        COALESCE(r.repete_mesmo_orgao_3anos,    0)  AS repete_mesmo_orgao_3anos

    FROM supplier_base b
    LEFT JOIN exclusividade  e USING (supplier_sk)
    LEFT JOIN repeticao_agg  r USING (supplier_sk)
""")

n = con.execute("SELECT COUNT(*) FROM feat_gF_rede").fetchone()[0]
print(f"feat_gF_rede: {n:,} rows")

print(con.execute("""
    SELECT
        SUM(fornecedor_exclusivo_orgao)             AS exclusivos,
        SUM(repete_mesmo_orgao_3anos)               AS repete_3anos,
        ROUND(AVG(qtd_orgaos_distintos_total), 2)   AS avg_orgaos,
        MAX(max_anos_mesmo_orgao)                   AS max_anos_orgao
    FROM feat_gF_rede
""").df().to_string(index=False))

In [ ]:
## Step 14 — Group G: Sectoral risk features
# Taxa de sanção agregada por setor — sem leakage individual
# Calculada a partir de dim_fornecedor (is_current = true)
# Fornece sinal de risco setorial sem expor sanção individual

con.execute("""
    CREATE OR REPLACE TABLE feat_gG_risco_setorial AS
    WITH base_cnae AS (
        SELECT
            b.supplier_sk,
            b.porte,
            b.uf,
            g4.cnae_divisao,
            b.tem_sancao
        FROM supplier_base b
        LEFT JOIN feat_g4_cadastral g4 USING (supplier_sk)
    ),
    taxa_cnae AS (
        SELECT
            cnae_divisao,
            ROUND(AVG(tem_sancao::INT) , 6)     AS taxa_sancao_cnae
        FROM base_cnae
        WHERE cnae_divisao IS NOT NULL
        GROUP BY cnae_divisao
    ),
    taxa_uf AS (
        SELECT
            uf,
            ROUND(AVG(tem_sancao::INT), 6)      AS taxa_sancao_uf
        FROM base_cnae
        GROUP BY uf
    ),
    taxa_porte AS (
        SELECT
            porte,
            ROUND(AVG(tem_sancao::INT), 6)      AS taxa_sancao_porte
        FROM base_cnae
        GROUP BY porte
    )
    SELECT
        b.supplier_sk,
        COALESCE(tc.taxa_sancao_cnae,   0.0)    AS taxa_sancao_cnae,
        COALESCE(tu.taxa_sancao_uf,     0.0)    AS taxa_sancao_uf,
        COALESCE(tp.taxa_sancao_porte,  0.0)    AS taxa_sancao_porte

    FROM supplier_base b
    LEFT JOIN feat_g4_cadastral g4 USING (supplier_sk)
    LEFT JOIN taxa_cnae  tc ON g4.cnae_divisao = tc.cnae_divisao
    LEFT JOIN taxa_uf    tu ON b.uf            = tu.uf
    LEFT JOIN taxa_porte tp ON b.porte         = tp.porte
""")

n = con.execute("SELECT COUNT(*) FROM feat_gG_risco_setorial").fetchone()[0]
print(f"feat_gG_risco_setorial: {n:,} rows")

print(con.execute("""
    SELECT
        ROUND(AVG(taxa_sancao_cnae),  6)    AS avg_taxa_cnae,
        ROUND(MAX(taxa_sancao_cnae),  6)    AS max_taxa_cnae,
        ROUND(AVG(taxa_sancao_uf),    6)    AS avg_taxa_uf,
        ROUND(MAX(taxa_sancao_uf),    6)    AS max_taxa_uf,
        ROUND(AVG(taxa_sancao_porte), 6)    AS avg_taxa_porte,
        ROUND(MAX(taxa_sancao_porte), 6)    AS max_taxa_porte
    FROM feat_gG_risco_setorial
""").df().to_string(index=False))

In [ ]:
## Step 15 — Assemble final feature table v3

con.execute(f"""
    COPY (
        SELECT
            -- chave
            b.supplier_sk,

            -- identidade base (4 categóricas)
            b.porte,
            b.natureza_juridica_desc,
            b.uf,
            b.situacao_cadastral,

            -- v1: contrato agregado (8 numéricas)
            COALESCE(fc.qtd_contratos,           0)      AS qtd_contratos,
            COALESCE(fc.qtd_orgaos_distintos,    0)      AS qtd_orgaos_distintos,
            COALESCE(CAST(fc.valor_total_contratos AS DOUBLE), 0.0) AS valor_total_contratos,
            COALESCE(fc.valor_medio_contrato,    0.0)    AS valor_medio_contrato,
            COALESCE(CAST(fc.valor_max_contrato AS DOUBLE), 0.0)    AS valor_max_contrato,
            COALESCE(fc.anos_ativo_contratos,    0)      AS anos_ativo_contratos,
            COALESCE(fc.tem_contrato_pncp,       0)      AS tem_contrato_pncp,
            COALESCE(fc.tem_contrato_compras,    0)      AS tem_contrato_compras,

            -- group 1: temporal acceleration (6)
            COALESCE(g1.contratos_2023,          0)      AS contratos_2023,
            COALESCE(g1.contratos_2024,          0)      AS contratos_2024,
            COALESCE(g1.contratos_2025,          0)      AS contratos_2025,
            COALESCE(g1.taxa_crescimento_2023_2024, 0.0) AS taxa_crescimento_2023_2024,
            COALESCE(g1.meses_desde_primeiro_contrato, 0) AS meses_desde_primeiro_contrato,
            COALESCE(g1.meses_desde_ultimo_contrato, 999) AS meses_desde_ultimo_contrato,

            -- group 2: concentration HHI (4)
            COALESCE(g2.hhi_orgaos,              0.0)    AS hhi_orgaos,
            COALESCE(g2.qtd_orgaos_unicos,       0)      AS qtd_orgaos_unicos,
            COALESCE(g2.pct_contratos_top1_orgao, 0.0)   AS pct_contratos_top1_orgao,
            COALESCE(g2.pct_valor_top1_orgao_valor, 0.0) AS pct_valor_top1_orgao_valor,

            -- group 3: value profile (5)
            COALESCE(g3.ratio_max_sobre_medio,   0.0)    AS ratio_max_sobre_medio,
            COALESCE(g3.qtd_contratos_alto_valor, 0)     AS qtd_contratos_alto_valor,
            COALESCE(g3.qtd_contratos_muito_alto_valor, 0) AS qtd_contratos_muito_alto_valor,
            COALESCE(g3.pct_contratos_pncp,      0.0)    AS pct_contratos_pncp,
            COALESCE(g3.pct_contratos_valor_zero, 0.0)   AS pct_contratos_valor_zero,

            -- group 4: cadastral basic (4 — 3 numéricas + 1 categórica)
            COALESCE(g4.idade_empresa_anos,      0)      AS idade_empresa_anos,
            g4.cnae_divisao,
            COALESCE(g4.capital_social,          0.0)    AS capital_social,
            COALESCE(g4.empresa_jovem,           0)      AS empresa_jovem,

            -- group A: contract modality/duration/retification (10)
            COALESCE(gA.pct_dispensa,            0.0)    AS pct_dispensa,
            COALESCE(gA.pct_pregao,              0.0)    AS pct_pregao,
            COALESCE(gA.pct_inexigibilidade,     0.0)    AS pct_inexigibilidade,
            COALESCE(gA.duracao_media_dias,      0.0)    AS duracao_media_dias,
            COALESCE(gA.duracao_max_dias,        0)      AS duracao_max_dias,
            COALESCE(gA.pct_empenho,             0.0)    AS pct_empenho,
            COALESCE(gA.pct_contrato_formal,     0.0)    AS pct_contrato_formal,
            COALESCE(gA.qtd_retificacoes,        0)      AS qtd_retificacoes,
            COALESCE(gA.tem_retificacao,         0)      AS tem_retificacao,

            -- group B: buyer sphere/power/geography (8)
            COALESCE(gB.pct_esfera_federal,      0.0)    AS pct_esfera_federal,
            COALESCE(gB.pct_esfera_estadual,     0.0)    AS pct_esfera_estadual,
            COALESCE(gB.pct_esfera_municipal,    0.0)    AS pct_esfera_municipal,
            COALESCE(gB.pct_poder_executivo,     0.0)    AS pct_poder_executivo,
            COALESCE(gB.pct_poder_legislativo,   0.0)    AS pct_poder_legislativo,
            COALESCE(gB.pct_poder_judiciario,    0.0)    AS pct_poder_judiciario,
            COALESCE(gB.qtd_ufs_compradoras,     0)      AS qtd_ufs_compradoras,
            COALESCE(gB.tem_comprador_externo,   0)      AS tem_comprador_externo,

            -- group C: cadastral advanced (4)
            COALESCE(gC.is_simples,              0)      AS is_simples,
            COALESCE(gC.is_mei,                  0)      AS is_mei,
            COALESCE(gC.qtd_cnaes_secundarios,   0)      AS qtd_cnaes_secundarios,
            COALESCE(gC.ratio_valor_sobre_capital, 0.0)  AS ratio_valor_sobre_capital,

            -- group D: value anomalies (4)
            COALESCE(gD.std_valor_contratos,     0.0)    AS std_valor_contratos,
            COALESCE(gD.cv_valor_contratos,      0.0)    AS cv_valor_contratos,
            COALESCE(gD.qtd_contratos_valor_1real, 0)    AS qtd_contratos_valor_1real,
            COALESCE(gD.gap_max_vs_medio,        0.0)    AS gap_max_vs_medio,

            -- group E: temporal advanced (5)
            COALESCE(gE.meses_entre_contratos_avg, 0.0)  AS meses_entre_contratos_avg,
            COALESCE(gE.dias_entre_contratos_min,  0)    AS dias_entre_contratos_min,
            COALESCE(gE.qtd_contratos_fds,         0)    AS qtd_contratos_fds,
            COALESCE(gE.pct_contratos_fds,         0.0)  AS pct_contratos_fds,
            COALESCE(gE.max_contratos_mesmo_mes,   0)    AS max_contratos_mesmo_mes,

            -- group F: network (4)
            COALESCE(gF.fornecedor_exclusivo_orgao, 0)   AS fornecedor_exclusivo_orgao,
            COALESCE(gF.qtd_orgaos_distintos_total, 0)   AS qtd_orgaos_distintos_total,
            COALESCE(gF.max_anos_mesmo_orgao,       0)   AS max_anos_mesmo_orgao,
            COALESCE(gF.repete_mesmo_orgao_3anos,   0)   AS repete_mesmo_orgao_3anos,

            -- group G: sectoral risk (3)
            COALESCE(gG.taxa_sancao_cnae,        0.0)    AS taxa_sancao_cnae,
            COALESCE(gG.taxa_sancao_uf,          0.0)    AS taxa_sancao_uf,
            COALESCE(gG.taxa_sancao_porte,       0.0)    AS taxa_sancao_porte,

            -- target
            b.tem_sancao,

            -- metadata
            CURRENT_TIMESTAMP                            AS _loaded_at

        FROM supplier_base b

        -- v1 base aggregates
        LEFT JOIN (
            SELECT
                supplier_sk,
                COUNT(*)                                         AS qtd_contratos,
                COUNT(DISTINCT codigo_orgao)                     AS qtd_orgaos_distintos,
                SUM(CASE WHEN NOT _valor_negativo THEN valor_inicial ELSE 0 END) AS valor_total_contratos,
                AVG(CASE WHEN NOT _valor_negativo THEN valor_inicial END)        AS valor_medio_contrato,
                MAX(CASE WHEN NOT _valor_negativo THEN valor_inicial END)        AS valor_max_contrato,
                DATEDIFF('year', MIN(data_referencia), MAX(data_referencia))     AS anos_ativo_contratos,
                MAX(CASE WHEN fonte = 'pncp'        THEN 1 ELSE 0 END)          AS tem_contrato_pncp,
                MAX(CASE WHEN fonte = 'compras_gov' THEN 1 ELSE 0 END)          AS tem_contrato_compras
            FROM fato_contratos
            GROUP BY supplier_sk
        ) fc USING (supplier_sk)

        LEFT JOIN feat_g1_temporal       g1 USING (supplier_sk)
        LEFT JOIN feat_g2_concentracao   g2 USING (supplier_sk)
        LEFT JOIN feat_g3_valor          g3 USING (supplier_sk)
        LEFT JOIN feat_g4_cadastral      g4 USING (supplier_sk)
        LEFT JOIN feat_gA_contrato       gA USING (supplier_sk)
        LEFT JOIN feat_gB_esfera         gB USING (supplier_sk)
        LEFT JOIN feat_gC_cadastral_adv  gC USING (supplier_sk)
        LEFT JOIN feat_gD_anomalia_valor gD USING (supplier_sk)
        LEFT JOIN feat_gE_temporal_adv   gE USING (supplier_sk)
        LEFT JOIN feat_gF_rede           gF USING (supplier_sk)
        LEFT JOIN feat_gG_risco_setorial gG USING (supplier_sk)
    )
    TO '{OUTPUT_PARQUET}'
    (FORMAT PARQUET)
""")

print(f"Written : {OUTPUT_PARQUET}")
print(f"Size    : {Path(OUTPUT_PARQUET).stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
## Step 16 — Validation

con.execute(f"CREATE OR REPLACE VIEW features_v3 AS SELECT * FROM read_parquet('{OUTPUT_PARQUET}')")

cols  = [r[0] for r in con.execute("DESCRIBE features_v3").fetchall()]
ncols = len(cols)

checks = []

total = con.execute("SELECT COUNT(*) FROM features_v3").fetchone()[0]
checks.append(("row_count = 810921",            total == 810_921,   total))

distinct_sk = con.execute("SELECT COUNT(DISTINCT supplier_sk) FROM features_v3").fetchone()[0]
checks.append(("supplier_sk unique",            distinct_sk == total, distinct_sk))

pos = con.execute("SELECT COUNT(*) FROM features_v3 WHERE tem_sancao = true").fetchone()[0]
neg = con.execute("SELECT COUNT(*) FROM features_v3 WHERE tem_sancao = false").fetchone()[0]
checks.append(("positives = 9946",              pos == 9_946,       pos))
checks.append(("negatives = 800975",            neg == 800_975,     neg))

leakage = [c for c in ["qtd_sancoes","tem_ceis","tem_cnep",
                        "sancao_ativa","valor_total_multa"] if c in cols]
checks.append(("no leakage columns",            len(leakage) == 0,  leakage or "clean"))

checks.append(("cnpj_normalized not exposed",   "cnpj_normalized" not in cols, True))

null_bin = con.execute("""
    SELECT COUNT(*) FROM features_v3
    WHERE tem_sancao IS NULL
       OR tem_contrato_pncp IS NULL
       OR empresa_jovem IS NULL
       OR is_simples IS NULL
       OR tem_retificacao IS NULL
""").fetchone()[0]
checks.append(("no nulls in binary features",   null_bin == 0,      null_bin))

null_cnae = con.execute("SELECT COUNT(*) FROM features_v3 WHERE cnae_divisao IS NULL").fetchone()[0]
checks.append(("cnae_divisao nulls <= 3227",    null_cnae <= 3_227, null_cnae))

# grupos presentes
for col in ["taxa_crescimento_2023_2024", "hhi_orgaos", "ratio_max_sobre_medio",
            "idade_empresa_anos", "pct_dispensa", "pct_esfera_federal",
            "is_simples", "cv_valor_contratos", "pct_contratos_fds",
            "fornecedor_exclusivo_orgao", "taxa_sancao_cnae"]:
    checks.append((f"{col} present", col in cols, col in cols))

print(f"{'Check':<45} {'Status':<8} {'Value'}")
print("-" * 75)
all_pass = True
for name, passed, value in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"{name:<45} {status:<8} {value}")
print("-" * 75)
print("ALL CHECKS PASSED ✅" if all_pass else "⚠️  SOME CHECKS FAILED — review above")

print(f"\nTotal columns : {ncols}")
print(f"Columns       : {cols}")

In [ ]:
## Step 17 — Export CSV and stratified split for H2O

import pandas as pd
from sklearn.model_selection import train_test_split

# ── export CSV completo ───────────────────────────────────────────────────────
con.execute(f"""
    COPY (
        SELECT
            supplier_sk, porte, natureza_juridica_desc, uf, situacao_cadastral,
            qtd_contratos, qtd_orgaos_distintos, valor_total_contratos,
            valor_medio_contrato, valor_max_contrato, anos_ativo_contratos,
            tem_contrato_pncp, tem_contrato_compras,
            contratos_2023, contratos_2024, contratos_2025,
            taxa_crescimento_2023_2024, meses_desde_primeiro_contrato,
            meses_desde_ultimo_contrato,
            hhi_orgaos, qtd_orgaos_unicos, pct_contratos_top1_orgao,
            pct_valor_top1_orgao_valor,
            ratio_max_sobre_medio, qtd_contratos_alto_valor,
            qtd_contratos_muito_alto_valor, pct_contratos_pncp,
            pct_contratos_valor_zero,
            idade_empresa_anos, cnae_divisao, capital_social, empresa_jovem,
            pct_dispensa, pct_pregao, pct_inexigibilidade,
            duracao_media_dias, duracao_max_dias,
            pct_empenho, pct_contrato_formal,
            qtd_retificacoes, tem_retificacao,
            pct_esfera_federal, pct_esfera_estadual, pct_esfera_municipal,
            pct_poder_executivo, pct_poder_legislativo, pct_poder_judiciario,
            qtd_ufs_compradoras, tem_comprador_externo,
            is_simples, is_mei, qtd_cnaes_secundarios, ratio_valor_sobre_capital,
            std_valor_contratos, cv_valor_contratos,
            qtd_contratos_valor_1real, gap_max_vs_medio,
            meses_entre_contratos_avg, dias_entre_contratos_min,
            qtd_contratos_fds, pct_contratos_fds, max_contratos_mesmo_mes,
            fornecedor_exclusivo_orgao, qtd_orgaos_distintos_total,
            max_anos_mesmo_orgao, repete_mesmo_orgao_3anos,
            taxa_sancao_cnae, taxa_sancao_uf, taxa_sancao_porte,
            tem_sancao
            -- _loaded_at excluído: metadado de pipeline
        FROM read_parquet('{OUTPUT_PARQUET}')
    )
    TO '{OUTPUT_CSV}'
    (FORMAT CSV, HEADER true, QUOTE '"')
""")

print(f"Written : {OUTPUT_CSV}")
print(f"Size    : {Path(OUTPUT_CSV).stat().st_size / 1024 / 1024:.1f} MB")

# ── stratified split 80/20 ────────────────────────────────────────────────────
TRAIN_PATH = OUTPUT_CSV.replace(".csv", "_train.csv")
TEST_PATH  = OUTPUT_CSV.replace(".csv", "_test.csv")

df = pd.read_csv(OUTPUT_CSV, quoting=1)
X  = df.drop(columns=["tem_sancao"])
y  = df["tem_sancao"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

train_df = pd.concat([X_train, y_train], axis=1)
test_df  = pd.concat([X_test,  y_test],  axis=1)

train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH,   index=False)

print(f"\nTrain : {len(train_df):,} rows | positives: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Test  : {len(test_df):,} rows  | positives: {y_test.sum():,}  ({y_test.mean()*100:.2f}%)")
print(f"\nTrain file: {TRAIN_PATH}")
print(f"Test  file: {TEST_PATH}")